# Smart MCQ Solver

Roll number 23f3000717. The task is to rank the five options of a multiple-choice
question so that the correct one lands as high as possible; the leaderboard metric
is mAP@3.

The notebook walks through the milestones in order: classical baselines, transformer
embeddings and zero-shot NLI, retrieval-augmented prompting, a ranker written from
scratch in PyTorch, a LoRA fine-tune, ensembling, and finally the submission itself.

## 0. Environment setup (Milestone 0)

In [ ]:
import os
import re
import string

import numpy as np
import pandas as pd
import torch

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if USE_CUDA else "cpu"
TORCH_DTYPE = torch.float16 if USE_CUDA else torch.float32

In [ ]:
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The notebook still runs on CPU, just slower.")

print("Device:", DEVICE)

In [ ]:
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1. Load the data

In [ ]:
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"

train_raw = pd.read_csv(f"{DATA_DIR}/train.csv")
test_raw = pd.read_csv(f"{DATA_DIR}/test.csv")

options = ["A", "B", "C", "D", "E"]
text_cols = ["prompt"] + options

print("train:", train_raw.shape)
print("test :", test_raw.shape)

In [ ]:
train_raw.head()

In [ ]:
print(train_raw.info())
print("\nMissing values per column:")
print(train_raw.isnull().sum())

In [ ]:
print("Answer counts:")
print(train_raw["answer"].value_counts())
print("\nExact duplicate rows:", train_raw.duplicated().sum())

## 2. Exploratory analysis

The answer letter is mildly imbalanced (B and C are the most common). `train_raw.duplicated()`
reports zero exact duplicates only because the boilerplate prefix differs between copies of a
question. We look for **question-level** duplicates in Section 4, since those are what threaten
a clean validation split.

In [ ]:
import matplotlib.pyplot as plt

counts = train_raw["answer"].value_counts().reindex(options)

plt.figure(figsize=(5, 3))
plt.bar(counts.index, counts.values, color="#4C72B0")
plt.title("Answer distribution (train)")
plt.xlabel("Correct option")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
print((train_raw["answer"].value_counts(normalize=True).reindex(options) * 100).round(1))

## 3. Text cleaning and query extraction

Two small helpers are used throughout:

- `clean_text` lowercases, drops URLs and punctuation, and squeezes whitespace. It is applied
  only to the **classical** models (TF-IDF, Word2Vec, the from-scratch vocabulary). Transformer
  models are given the raw text they were pretrained on.
- `extract_query` strips the quiz boilerplate ("Pick the best answer:", "... carefully.") so we
  are left with the bare question stem. This is used both for retrieval queries and to identify
  duplicate questions.

In [ ]:
def clean_text(text):
    """Lowercase, remove URLs and punctuation, collapse whitespace."""
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
PROMPT_PREFIXES = [
    r"^pick the best possible answer\s*:?\s*",
    r"^select the most accurate option\s*:?\s*",
    r"^determine the correct option\s*:?\s*",
    r"^identify the correct statement\s*:?\s*",
    r"^choose the correct answer\s*:?\s*",
    r"^which of the following is correct\??\s*:?\s*",
    r"^which of the following statements is true (about|regarding)\s*:?\s*",
    r"^which of the following statements accurately (describes|depicts)\s*:?\s*",
]

PROMPT_SUFFIXES = [
    r"\s*among the listed options\.?$",
    r"\s*from the following choices\.?$",
    r"\s*based on the given context\.?$",
    r"\s*carefully\.?$",
]

In [ ]:
def extract_query(prompt):
    """Remove quiz boilerplate, leaving the question stem."""
    text = str(prompt).strip()
    for pattern in PROMPT_PREFIXES:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    for pattern in PROMPT_SUFFIXES:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    return text.strip()

In [ ]:
for prompt in train_raw["prompt"].head(3):
    print(repr(prompt[:70]))
    print("  ->", repr(extract_query(prompt)[:70]))

## 4. Leak-safe train / validation split

Duplication in this dataset happens at **two levels**:

1. **Boilerplate copies** - the same stem and the same five options, wrapped in different quiz
   phrases. These share a full question key (stem + options).
2. **Paraphrased variants** - the same stem, but the options are lightly reworded ("mechanism"
   vs "framework", "principle" vs "concept"). These have *different* full keys, so a split on
   stem + options still lets a variant of a training question land in validation. A model that
   memorises answer wording then scores far too high on validation - exactly the inflated
   validation / weak public score gap we saw in earlier runs.

The fix: the split unit is the **question stem alone**. All boilerplate copies *and* all
paraphrased variants of a stem stay on the same side. We keep one row per unique question for
training, but assign whole stem groups to either train or validation.

In [ ]:
def question_stem(row):
    """The bare question text. Paraphrased variants share a stem."""
    return clean_text(extract_query(row["prompt"]))


def question_key(row):
    """Full identity of a question: its stem plus its five options."""
    opts = "|".join(clean_text(row[opt]) for opt in options)
    return f"{question_stem(row)}||{opts}"


def add_question_keys(df):
    """Return a copy of df with `stem` and `qkey` columns attached."""
    out = df.copy()
    out["stem"] = out.apply(question_stem, axis=1)
    out["qkey"] = out.apply(question_key, axis=1)
    return out

In [ ]:
train_raw = add_question_keys(train_raw)

print(f"rows: {len(train_raw)}   unique questions: {train_raw['qkey'].nunique()}   "
      f"unique stems: {train_raw['stem'].nunique()}")

In [ ]:
repeated = train_raw["qkey"].value_counts()
repeated = repeated[repeated > 1]

print("Boilerplate copies of one question:")
for _, row in train_raw[train_raw["qkey"] == repeated.index[0]].head(4).iterrows():
    print(f"  id={row['id']:>4}  answer={row['answer']}  prompt={row['prompt'][:70]!r}")

In [ ]:
per_stem = train_raw.groupby("stem")["qkey"].nunique()
print("stems with more than one paraphrased option-set:", int((per_stem > 1).sum()))

example_stem = per_stem[per_stem > 1].index[0]
print(f"\nExample stem: {example_stem[:70]!r}")
for _, row in train_raw[train_raw["stem"] == example_stem].drop_duplicates("qkey").head(2).iterrows():
    print(f"  answer={row['answer']}  option A = {clean_text(row['A'])[:70]!r}")

In [ ]:
from sklearn.model_selection import train_test_split

train_unique = train_raw.drop_duplicates(subset="qkey", keep="first").reset_index(drop=True)

stems = train_unique.groupby("stem", as_index=False).agg(answer=("answer", "first"))
tr_stems, val_stems = train_test_split(
    stems["stem"], test_size=0.2, random_state=SEED, stratify=stems["answer"],
)

tr = train_unique[train_unique["stem"].isin(set(tr_stems))].reset_index(drop=True)
val = train_unique[train_unique["stem"].isin(set(val_stems))].reset_index(drop=True)

assert set(tr["stem"]).isdisjoint(set(val["stem"])), "train/val leak: shared stem found"
print("train:", tr.shape, " validation:", val.shape)
print("split is disjoint at the stem level")

In [ ]:
tr_clean = tr.copy()
val_clean = val.copy()
for col in text_cols:
    tr_clean[col] = tr_clean[col].apply(clean_text)
    val_clean[col] = val_clean[col].apply(clean_text)

print("answer proportions  tr / val:")
print(pd.concat(
    [tr["answer"].value_counts(normalize=True).rename("tr"),
     val["answer"].value_counts(normalize=True).rename("val")],
    axis=1).reindex(options).round(3))

## 5. Evaluation metrics

The competition uses **mAP@3**. For a single question with one correct answer, average precision
at 3 is `1 / rank` if the answer is in the top three, else 0. We also track **accuracy** and
**macro-F1** on the top-1 choice so every model logged to W&B is comparable on the metrics the
project guidelines ask for.

Every model in this notebook ends up producing an `(n_questions, 5)` score matrix, so a single
helper turns scores into the `"A B C"` strings the leaderboard expects.

In [ ]:
def apk(actual, predicted, k=3):
    """Average precision at k for a question with a single correct answer."""
    for i, letter in enumerate(predicted[:k]):
        if letter == actual:
            return 1.0 / (i + 1)
    return 0.0


def mapk(actuals, predictions, k=3):
    """Mean average precision at k over 'A B C' style prediction strings."""
    return float(np.mean([apk(a, p.split(), k) for a, p in zip(actuals, predictions)]))

In [ ]:
assert apk("A", ["A", "B", "C"]) == 1.0
assert apk("B", ["A", "B", "C"]) == 0.5
assert apk("D", ["A", "B", "C"]) == 0.0
print("mAP@3 helpers OK")

In [ ]:
def top3_from_scores(scores):
    """Turn an (n, 5) score matrix into 'A B C' predictions, highest score first.

    The sort is stable, so exactly tied options always fall back to A-E order
    instead of whatever the sorting algorithm happens to do.
    """
    letters = np.array(options)
    order = np.argsort(-np.asarray(scores, dtype=float), axis=1, kind="stable")
    return [" ".join(letters[row][:3]) for row in order]

In [ ]:
from sklearn.metrics import accuracy_score, f1_score


def eval_all(actuals, predictions_top3):
    """mAP@3, accuracy and macro-F1 for a list of 'A B C' predictions."""
    actuals = list(actuals)
    top1 = [p.split()[0] for p in predictions_top3]
    return {
        "map3": mapk(actuals, predictions_top3),
        "accuracy": float(accuracy_score(actuals, top1)),
        "macro_f1": float(f1_score(actuals, top1, average="macro",
                                   labels=options, zero_division=0)),
    }


print(eval_all(["A", "B"], ["A B C", "C A B"]))

## 6. Weights & Biases tracking (Milestone 0)

Every model reports its three metrics to the same project through `record`, which also appends
to a shared `results` table used for the comparison charts. The API key is set directly in the
notebook - this is a private notebook, so the key is not shared.

In [ ]:
import wandb

WANDB_API_KEY = "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"
WANDB_ENTITY = "23f3000717-dl-genai-project-"
WANDB_PROJECT = "23f3000717-dl-genai-project"

WANDB_ENABLED = False
try:
    wandb.login(key=WANDB_API_KEY)
    WANDB_ENABLED = True
    print("W&B ready. Project:", WANDB_PROJECT)
except Exception as error:
    print("W&B login failed; metrics will still print locally:", error)

In [ ]:
def log_run(model_name, metrics, extra_config=None):
    """Log one model as a single W&B run."""
    print(f"{model_name:34s} " + "  ".join(f"{k}={v:.4f}" for k, v in metrics.items()))
    if not WANDB_ENABLED:
        return
    config = {"model": model_name, "k": 3, "seed": SEED}
    if extra_config:
        config.update(extra_config)
    if wandb.run is not None:
        wandb.run.finish()
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=model_name, config=config)
    wandb.log(metrics)
    run.finish()

In [ ]:
results = []


def record(name, metrics, extra_config=None):
    """Store a model's metrics in the results table and log them to W&B."""
    results.append({"Model": name, **metrics})
    log_run(name, metrics, extra_config)


def leaderboard():
    """The results collected so far, best mAP@3 first."""
    return pd.DataFrame(results).sort_values("map3", ascending=False).reset_index(drop=True).round(4)

## 7. Baseline models (Milestone 1)

### 7.1 Random baseline

With five options and a top-3 guess, a random ranking already scores around 0.32 on mAP@3 purely
by chance. Any real method has to beat this bar.

In [ ]:
rng = np.random.default_rng(SEED)
random_preds = [" ".join(rng.permutation(options)[:3]) for _ in range(len(val))]

record("Random baseline", eval_all(val["answer"], random_preds))

### 7.2 TF-IDF and cosine similarity

The vectorizer is fitted on the **training** text only. Each option is scored by the cosine
similarity between its TF-IDF vector and the prompt's, then the options are ranked.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf.fit(pd.concat([tr_clean[col] for col in text_cols]))

print("Vocabulary size:", len(tfidf.vocabulary_))

In [ ]:
def tfidf_scores(df, vectorizer):
    """(n_questions, 5) cosine similarity between each option and its prompt."""
    prompt_mat = vectorizer.transform(df["prompt"])
    return np.column_stack([
        cosine_similarity(prompt_mat, vectorizer.transform(df[opt])).diagonal()
        for opt in options
    ])

In [ ]:
val_preds_tfidf = top3_from_scores(tfidf_scores(val_clean, tfidf))
record("TF-IDF", eval_all(val["answer"], val_preds_tfidf))

In [ ]:
for i in range(5):
    print("Prompt    :", val.loc[i, "prompt"][:90])
    print("Prediction:", val_preds_tfidf[i], "| Actual:", val.loc[i, "answer"])
    print()

### 7.3 Word2Vec embeddings

A small Word2Vec model is trained on the cleaned training text. A sentence is represented by the
mean of its word vectors, and options are ranked by cosine similarity to the prompt.

In [ ]:
from gensim.models import Word2Vec

sentences = []
for col in text_cols:
    sentences.extend(tr_clean[col].apply(lambda t: str(t).split()).tolist())
print("Sentences for Word2Vec:", len(sentences))

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4, seed=SEED)
print("Vocab size:", len(w2v_model.wv))

In [ ]:
def sentence_vector(text, model):
    """Mean of the word vectors present in the model's vocabulary."""
    vectors = [model.wv[word] for word in str(text).split() if word in model.wv]
    if not vectors:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)


def w2v_scores(df, model):
    """(n_questions, 5) cosine similarity between option and prompt sentence vectors."""
    prompt_vecs = np.array([sentence_vector(p, model) for p in df["prompt"]])
    return np.column_stack([
        cosine_similarity(
            prompt_vecs, np.array([sentence_vector(t, model) for t in df[opt]])
        ).diagonal()
        for opt in options
    ])

In [ ]:
val_preds_w2v = top3_from_scores(w2v_scores(val_clean, w2v_model))
record("Word2Vec", eval_all(val["answer"], val_preds_w2v))

In [ ]:
for word in ["quantum", "galaxy", "entropy"]:
    if word in w2v_model.wv:
        print(word, "->", [w for w, _ in w2v_model.wv.most_similar(word, topn=5)])

## 8. Transformers: BERT, RoBERTa and attention (Milestone 2)

**BERT** learns contextual word representations by reading text in both directions. **RoBERTa**
is a more heavily trained variant that drops BERT's next-sentence objective. Both rely on
**self-attention**, which lets every token weigh every other token, capturing long-range context.
Unlike Word2Vec, these models give **context-aware** embeddings: the same word gets a different
vector depending on its surroundings.

We start with MiniLM, a compact sentence-embedding model, and score options by cosine similarity
to the prompt embedding.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

In [ ]:
from transformers import AutoTokenizer

minilm_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

sample_ids = minilm_tokenizer(val["prompt"].iloc[0], truncation=True, max_length=64)["input_ids"]
print("Token ids:", sample_ids[:12])
print("Decoded  :", minilm_tokenizer.decode(sample_ids))

In [ ]:
lengths = [len(minilm_tokenizer(p, truncation=True, max_length=64)["input_ids"])
           for p in val["prompt"]]
print("Prompt token length min/mean/max:",
      min(lengths), round(sum(lengths) / len(lengths), 1), max(lengths))

In [ ]:
def embedding_scores(df, model, batch_size=64):
    """(n_questions, 5) cosine similarity between option and prompt embeddings."""
    def encode(texts):
        return model.encode(list(texts), convert_to_numpy=True,
                            batch_size=batch_size, show_progress_bar=False)

    prompt_emb = encode(df["prompt"])
    return np.column_stack([
        cosine_similarity(prompt_emb, encode(df[opt])).diagonal() for opt in options
    ])

In [ ]:
cos_val_scores = embedding_scores(val, minilm)
val_preds_minilm = top3_from_scores(cos_val_scores)

record("MiniLM", eval_all(val["answer"], val_preds_minilm))

In [ ]:
leaderboard()

## 9. Zero-shot classification with NLI (Milestone 2)

Pure similarity does not tell us which option is *correct*, because all five distractors share the
prompt's vocabulary. A natural-language-inference model reframes the task: treat the question stem
as a premise and each option as a hypothesis, and read off the model's **entailment** score.
We use `MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli`, a DeBERTa-v3-large fine-tuned
on five NLI datasets.

The hypothesis formulation matters a lot: passing the option text directly as the hypothesis
scores far better than wrapping it in a template sentence like "This example is ..."
(validation mAP@3 0.66 vs 0.53).

The forward pass returns all three NLI logits and we keep them, because Section 14 needs the
contradiction head as well as the entailment head.

In [ ]:
from transformers import AutoModelForSequenceClassification

NLI_MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME, dtype=TORCH_DTYPE
).to(DEVICE).eval()

In [ ]:
label2id = {str(k).lower(): v for k, v in nli_model.config.label2id.items()}
ENTAILMENT_ID = label2id["entailment"]
CONTRADICTION_ID = label2id["contradiction"]

print("label ids:", nli_model.config.label2id)

In [ ]:
@torch.no_grad()
def nli_logits(premises, hypotheses, batch_size=64, max_length=256, sort_by_length=False):
    """Raw NLI logits for each (premise, hypothesis) pair.

    The precision comes from the model's own dtype, so casting the model to
    float32 is enough to get a float32 pass. `sort_by_length` puts pairs of
    similar length in the same batch, which changes how much padding each
    batch carries and therefore its exact arithmetic.
    """
    lengths = [len(p) + len(h) for p, h in zip(premises, hypotheses)]
    order = np.argsort(lengths, kind="stable") if sort_by_length else np.arange(len(premises))

    out = None
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        encoded = nli_tokenizer(
            [premises[i] for i in idx], [hypotheses[i] for i in idx],
            return_tensors="pt", truncation=True, max_length=max_length, padding=True,
        ).to(DEVICE)
        logits = nli_model(**encoded).logits.float().cpu().numpy()
        if out is None:
            out = np.empty((len(premises), logits.shape[1]), dtype=np.float32)
        out[idx] = logits
    return out

In [ ]:
def nli_option_logits(df, prompts=None, **kwargs):
    """(n_questions, 5, n_labels) logits: premise = question stem, hypothesis = option."""
    df = df.reset_index(drop=True)
    premise_text = [extract_query(p) for p in df["prompt"]] if prompts is None else list(prompts)

    premises, hypotheses = [], []
    for i, row in df.iterrows():
        for opt in options:
            premises.append(premise_text[i])
            hypotheses.append(str(row[opt]))

    logits = nli_logits(premises, hypotheses, **kwargs)
    return logits.reshape(len(df), len(options), -1)

In [ ]:
def entail_score(logits):
    """Rank options by the entailment logit alone."""
    return logits[..., ENTAILMENT_ID]


def entail_minus_contra(logits):
    """Rank options by entailment against contradiction."""
    return logits[..., ENTAILMENT_ID] - logits[..., CONTRADICTION_ID]

In [ ]:
nli_val_logits = nli_option_logits(val)
nli_val_scores = entail_score(nli_val_logits)
val_preds_nli = top3_from_scores(nli_val_scores)

record("Zero-Shot NLI (DeBERTa-v3-large)", eval_all(val["answer"], val_preds_nli),
       extra_config={"nli_model": NLI_MODEL_NAME})

## 10. Retrieval-Augmented Generation (Milestone 3)

### Why RAG

The zero-shot model answers only from what is baked into its weights. When a question hinges on a
specific fact it never learned, it can only guess. RAG supplies relevant text at inference time:
**retrieve** passages related to the question, **augment** the prompt with them, then **predict**.

### An offline, pre-built vector store

Rather than calling the Wikipedia API at run time (slow, rate-limited, non-deterministic), we build
the vector store **once** from a corpus we already have: the option statements in the training
split. These are short factual sentences about the same topics the questions cover. The FAISS index
is saved to disk and reloaded on later runs, so retrieval needs no network.

Because the store is built from the **training** questions only, and validation questions are
disjoint from training (Section 4), retrieval cannot hand the model its own answer.

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import pickle

import faiss

VECTOR_DB_DIR = "/kaggle/working/vector_db"
os.makedirs(VECTOR_DB_DIR, exist_ok=True)


def build_corpus(df):
    """The knowledge base: unique option statements from a dataframe."""
    passages = []
    for opt in options:
        passages.extend(df[opt].astype(str).tolist())
    return list(dict.fromkeys(p.strip() for p in passages if p.strip()))

In [ ]:
def build_or_load_index(passages, encoder, name):
    """Build a FAISS index once and persist it; reload it on subsequent runs."""
    index_path = f"{VECTOR_DB_DIR}/{name}.index"
    meta_path = f"{VECTOR_DB_DIR}/{name}.pkl"

    if os.path.exists(index_path) and os.path.exists(meta_path):
        print(f"Loading pre-built vector store '{name}'")
        with open(meta_path, "rb") as handle:
            return faiss.read_index(index_path), pickle.load(handle)

    print(f"Building vector store '{name}' from {len(passages)} passages")
    emb = encoder.encode(passages, convert_to_numpy=True, batch_size=64,
                         show_progress_bar=False, normalize_embeddings=True)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)

    faiss.write_index(index, index_path)
    with open(meta_path, "wb") as handle:
        pickle.dump(passages, handle)
    print("Passages indexed:", index.ntotal)
    return index, passages

In [ ]:
corpus_passages = build_corpus(tr)
vector_index, passage_texts = build_or_load_index(corpus_passages, minilm, name="train_corpus")

print("Vector store size:", vector_index.ntotal)

In [ ]:
def retrieve_context(prompts, encoder, index, passages, k=5, batch_size=64):
    """The top-k stored passages most similar to each question stem."""
    queries = [extract_query(p) for p in prompts]
    query_emb = encoder.encode(queries, convert_to_numpy=True, batch_size=batch_size,
                               show_progress_bar=False, normalize_embeddings=True)
    _, idxs = index.search(query_emb, min(k, index.ntotal))
    return [" ".join(passages[i] for i in row if i != -1) for row in idxs]

In [ ]:
sample_prompt = val.loc[0, "prompt"]
sample_context = retrieve_context([sample_prompt], minilm, vector_index, passage_texts)[0]

print("PROMPT :", sample_prompt)
print("CONTEXT:", sample_context[:400], "...")

In [ ]:
def make_rag_prompts(df, encoder, index, passages, k=5, max_context_chars=600):
    """Prepend retrieved context to each question, forming the augmented premise."""
    contexts = retrieve_context(df["prompt"].tolist(), encoder, index, passages, k=k)
    prompts = []
    for context, (_, row) in zip(contexts, df.iterrows()):
        choices = "\n".join(f"{o}) {row[o]}" for o in options if pd.notna(row[o]))
        question = f"Question: {row['prompt']}\nChoices:\n{choices}"
        prompts.append(f"Context: {context[:max_context_chars]}\n{question}" if context else question)
    return prompts

In [ ]:
val_rag_prompts = make_rag_prompts(val, minilm, vector_index, passage_texts)
val_preds_rag = top3_from_scores(entail_score(nli_option_logits(val, prompts=val_rag_prompts)))

record("Zero-Shot + RAG", eval_all(val["answer"], val_preds_rag),
       extra_config={"retriever": "MiniLM+FAISS", "k": 5})

In [ ]:
leaderboard()

## 11. A model built from scratch: BiLSTM ranker

This model is written end to end with no pretrained weights and no Hugging Face components: its
own vocabulary, its own `Dataset` and padding `collate`, its own architecture, and a hand-written
training loop (forward, loss, backward, step, zero-grad).

Each question is exploded into five `(question, option)` pairs and the model performs binary
classification: does this option answer this question? The probability of the "correct" class
becomes the option's score, and the five scores are ranked into a top-3 prediction. We keep the
checkpoint with the best **validation** mAP@3 to avoid overfitting the training set.

In [ ]:
def build_pairs(df, with_labels=True):
    """Explode an MCQ dataframe into one row per (question, option).

    `example_id` and `option` let us regroup the five rows per question at inference
    time. Shared by the from-scratch model and the LoRA model.
    """
    texts, labels, example_ids, opt_letters = [], [], [], []
    for _, row in df.reset_index(drop=True).iterrows():
        question = extract_query(row["prompt"])
        for opt in options:
            texts.append(f"Question: {question}\nOption: {row[opt]}")
            example_ids.append(int(row["id"]))
            opt_letters.append(opt)
            if with_labels:
                labels.append(1 if row["answer"] == opt else 0)

    pairs = {"text": texts, "example_id": example_ids, "option": opt_letters}
    if with_labels:
        pairs["label"] = labels
    return pairs

In [ ]:
train_pairs = build_pairs(tr)
val_pairs = build_pairs(val)

print("train pairs:", len(train_pairs["text"]), "=", len(tr), "questions x 5")
print("val   pairs:", len(val_pairs["text"]), "=", len(val), "questions x 5")
print("label balance (train):", np.bincount(train_pairs["label"]), "-> 1-in-5 positive")

In [ ]:
from collections import Counter

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset

PAD_ID, UNK_ID = 0, 1


def build_vocab(texts, min_freq=2, max_size=20000):
    """Vocabulary built only from training text, so nothing leaks from validation."""
    counter = Counter()
    for text in texts:
        counter.update(clean_text(text).split())
    itos = ["<pad>", "<unk>"] + [w for w, f in counter.most_common(max_size) if f >= min_freq]
    return {w: i for i, w in enumerate(itos)}, itos

In [ ]:
stoi, itos = build_vocab(train_pairs["text"])
print("Vocab size:", len(itos))


def encode_text(text, max_len=220):
    """Map a string to a list of vocabulary ids, truncated to `max_len` tokens."""
    ids = [stoi.get(word, UNK_ID) for word in clean_text(text).split()[:max_len]]
    return ids or [UNK_ID]

In [ ]:
class PairDataset(TorchDataset):
    """One (question, option) pair per item, encoded on demand."""

    def __init__(self, pairs, with_labels=True):
        self.texts = pairs["text"]
        self.labels = pairs["label"] if with_labels else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        ids = torch.tensor(encode_text(self.texts[i]), dtype=torch.long)
        label = self.labels[i] if self.labels is not None else 0
        return ids, torch.tensor(label, dtype=torch.long)

In [ ]:
def collate(batch):
    """Pad each batch to its own longest sequence."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    padded = torch.full((len(seqs), int(lengths.max())), PAD_ID, dtype=torch.long)
    for i, seq in enumerate(seqs):
        padded[i, :len(seq)] = seq
    return padded, lengths, torch.stack(labels)

In [ ]:
scratch_train_ds = PairDataset(train_pairs)
scratch_val_ds = PairDataset(val_pairs)

scratch_train_loader = DataLoader(scratch_train_ds, batch_size=64, shuffle=True, collate_fn=collate)
scratch_val_loader = DataLoader(scratch_val_ds, batch_size=128, shuffle=False, collate_fn=collate)

print("batches per epoch:", len(scratch_train_loader))

In [ ]:
class BiLSTMRanker(nn.Module):
    """Embedding -> BiLSTM -> concat(mean-pool, max-pool) -> dropout -> linear.

    Padding positions are masked out of both pools so short options are not diluted.
    """

    def __init__(self, vocab_size, emb_dim=128, hidden=128, num_classes=2, pad_id=PAD_ID):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden * 2 * 2, num_classes)

    def forward(self, x, lengths):
        mask = (x != PAD_ID).unsqueeze(-1)
        out, _ = self.lstm(self.emb(x))
        out = out.masked_fill(~mask, 0.0)
        mean_pool = out.sum(1) / lengths.clamp(min=1).unsqueeze(1).to(out.dtype)
        max_pool = out.masked_fill(~mask, -1e9).max(1).values
        return self.fc(self.dropout(torch.cat([mean_pool, max_pool], dim=1)))

In [ ]:
scratch_model = BiLSTMRanker(len(itos)).to(DEVICE)
print("Trainable params:", sum(p.numel() for p in scratch_model.parameters()))

In [ ]:
def scratch_scores(loader, model, n_questions):
    """(n_questions, 5) probability that each option is the correct one."""
    model.eval()
    probs = []
    with torch.no_grad():
        for x, lengths, _ in loader:
            logits = model(x.to(DEVICE), lengths.to(DEVICE))
            probs.extend(F.softmax(logits, dim=-1)[:, 1].cpu().numpy().tolist())
    return np.array(probs).reshape(n_questions, len(options))

In [ ]:
optimizer = torch.optim.Adam(scratch_model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 4.0], device=DEVICE))

SCRATCH_EPOCHS = 10

In [ ]:
best_map3, best_state = -1.0, None

for epoch in range(1, SCRATCH_EPOCHS + 1):
    scratch_model.train()
    running = 0.0
    for x, lengths, y in scratch_train_loader:
        x, lengths, y = x.to(DEVICE), lengths.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(scratch_model(x, lengths), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(scratch_model.parameters(), 1.0)
        optimizer.step()
        running += loss.item() * len(y)

    metrics = eval_all(val["answer"],
                       top3_from_scores(scratch_scores(scratch_val_loader, scratch_model, len(val))))
    print(f"epoch {epoch:2d} | loss {running / len(scratch_train_ds):.4f} "
          f"| map3 {metrics['map3']:.4f} | acc {metrics['accuracy']:.4f} "
          f"| f1 {metrics['macro_f1']:.4f}")

    if metrics["map3"] > best_map3:
        best_map3 = metrics["map3"]
        best_state = {k: v.detach().cpu().clone() for k, v in scratch_model.state_dict().items()}

scratch_model.load_state_dict(best_state)
print("\nBest validation mAP@3:", round(best_map3, 4))

In [ ]:
scratch_val_scores = scratch_scores(scratch_val_loader, scratch_model, len(val))
val_preds_scratch = top3_from_scores(scratch_val_scores)

record("From-scratch BiLSTM", eval_all(val["answer"], val_preds_scratch),
       extra_config={"arch": "BiLSTM+meanmax", "emb_dim": 128, "hidden": 128,
                     "epochs": SCRATCH_EPOCHS, "vocab": len(itos), "from_scratch": True})

In [ ]:
torch.save(scratch_model.state_dict(), "/kaggle/working/bilstm_scratch.pt")
print("Saved /kaggle/working/bilstm_scratch.pt")

## 12. LoRA fine-tuning of RoBERTa (Milestone 4)

### Formulating the MCQ task

We reuse the exploded `(question, option)` pairs from Section 11 and frame the problem as binary
classification. At inference we run all five pairs, take the probability of the "correct" class as
each option's score, and rank them.

### LoRA vs full fine-tuning

We fine-tune `roberta-base` - a stronger encoder than DistilBERT, as the Future-Work section
suggested - under a LoRA recipe (set `BASE_MODEL` back to `distilbert-base-uncased` if GPU memory
is tight). Full fine-tuning updates every weight (~125M for RoBERTa); the optimizer stores two
extra copies of each, so memory roughly triples, and small datasets risk catastrophic forgetting.
**LoRA** freezes the base model and trains tiny low-rank adapters in the attention and feed-forward
layers, so the update `W' = W + (alpha / r) * B A` touches only a few percent of the parameters.
Checkpoints are a few megabytes and the base weights stay intact.

In [ ]:
!pip install -q "transformers>=4.40" "peft>=0.11" "accelerate>=0.30" datasets sentencepiece

In [ ]:
from datasets import Dataset

BASE_MODEL = "roberta-base"
MAX_LENGTH = 256

ft_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def tokenize_batch(batch):
    return ft_tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


def to_hf_dataset(pairs):
    """Tokenized dataset holding only what the Trainer needs."""
    return (Dataset.from_dict(pairs)
            .map(tokenize_batch, batched=True, remove_columns=["text"])
            .remove_columns(["example_id", "option"]))

In [ ]:
ft_train_ds = to_hf_dataset(train_pairs)
ft_val_ds = to_hf_dataset(val_pairs)

print(ft_train_ds)
print("columns:", ft_train_ds.column_names)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value", "intermediate.dense", "output.dense"],
    bias="none",
)

base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
model = get_peft_model(base_model, lora_config).to(DEVICE)
model.print_trainable_parameters()

### Training loop, GPU memory and efficiency

The Hugging Face `Trainer` wraps the standard loop; the levers live in `TrainingArguments`:

- `per_device_train_batch_size` is the main memory dial; halve it on OOM.
- `gradient_accumulation_steps` keeps the *effective* batch large without the memory cost.
- `fp16` mixed precision roughly doubles throughput and halves activation memory.
- Dynamic padding avoids wasted compute on padding tokens.
- `load_best_model_at_end` with early stopping keeps the checkpoint that generalises best.

In [ ]:
from transformers import (DataCollatorWithPadding, EarlyStoppingCallback, Trainer,
                          TrainingArguments)

data_collator = DataCollatorWithPadding(tokenizer=ft_tokenizer)
val_answers = val["answer"].reset_index(drop=True)


def compute_metrics(eval_pred):
    """Regroup the flat pair predictions back into five options per question."""
    logits, _ = eval_pred
    assert len(logits) == len(val) * len(options), "eval alignment broken"
    correct_probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    scores = correct_probs.reshape(len(val), len(options))
    return eval_all(val_answers, top3_from_scores(scores))

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/lora_mcq",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=USE_CUDA,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    logging_steps=25,
    report_to="none",
    seed=SEED,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ft_train_ds,
    eval_dataset=ft_val_ds,
    processing_class=ft_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
trainer.train()
if USE_CUDA:
    print("Peak GPU memory:", round(torch.cuda.max_memory_allocated() / 1e9, 2), "GB")

The `Trainer` renders its progress table as a widget, which does not survive when the notebook is
saved. Printing the log history keeps the per-epoch curve in the notebook: a validation mAP@3 that
is still climbing at epoch 10 means undertrained, one that peaks early and falls means overfitting.

In [ ]:
history = pd.DataFrame([h for h in trainer.state.log_history if "eval_map3" in h])
print(history[["epoch", "eval_loss", "eval_map3", "eval_accuracy", "eval_macro_f1"]]
      .round(4).to_string(index=False))

In [ ]:
@torch.no_grad()
def finetuned_scores(df, model, tokenizer, batch_size=64, max_length=MAX_LENGTH):
    """(n_questions, 5) probability that each option is the correct one."""
    model.eval()
    texts = build_pairs(df, with_labels=False)["text"]
    probs = []
    for i in range(0, len(texts), batch_size):
        encoded = tokenizer(texts[i:i + batch_size], truncation=True, max_length=max_length,
                            padding=True, return_tensors="pt").to(DEVICE)
        probs.extend(F.softmax(model(**encoded).logits, dim=-1)[:, 1].cpu().numpy().tolist())
    return np.array(probs).reshape(len(df), len(options))

In [ ]:
ft_val_scores = finetuned_scores(val, model, ft_tokenizer)
val_preds_ft = top3_from_scores(ft_val_scores)

record(f"LoRA fine-tuned ({BASE_MODEL})", eval_all(val["answer"], val_preds_ft),
       extra_config={"base_model": BASE_MODEL, "lora_r": lora_config.r,
                     "lora_alpha": lora_config.lora_alpha,
                     "epochs": training_args.num_train_epochs,
                     "lr": training_args.learning_rate})

In [ ]:
ADAPTER_DIR = "/kaggle/working/lora_mcq_adapter"
model.save_pretrained(ADAPTER_DIR)
ft_tokenizer.save_pretrained(ADAPTER_DIR)

size_mb = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f))
              for f in os.listdir(ADAPTER_DIR)) / 1e6
print(f"Saved LoRA adapter ({size_mb:.1f} MB)")

In [ ]:
leaderboard()

## 13. Ensembling (Milestone 5)

Every model reduces a question to a per-option score vector, which is what makes ensembling
possible. We combine models with **rank averaging**: convert each model's five scores to ranks,
average the ranks (optionally weighted), and re-rank. Ranks are scale-free, so models with very
different score ranges combine cleanly.

In [ ]:
from scipy.stats import rankdata


def rank_average(*score_mats, weights=None):
    """Average the per-question ranks of several models and re-rank to a top-3."""
    ranks = [np.apply_along_axis(rankdata, 1, mat) for mat in score_mats]
    averaged = np.average(np.stack(ranks), axis=0, weights=weights)
    return top3_from_scores(averaged), averaged

In [ ]:
combos = {
    "Ensemble (LoRA + NLI)": ([ft_val_scores, nli_val_scores], None),
    "Ensemble (LoRA + NLI, 3:1)": ([ft_val_scores, nli_val_scores], [3, 1]),
    "Ensemble (LoRA + NLI + BiLSTM)": ([ft_val_scores, nli_val_scores, scratch_val_scores], [2, 1, 1]),
    "Ensemble (LoRA + NLI + cosine)": ([ft_val_scores, nli_val_scores, cos_val_scores], [3, 1, 1]),
}

ensemble_table = []
for name, (mats, weights) in combos.items():
    preds, _ = rank_average(*mats, weights=weights)
    ensemble_table.append({"Ensemble": name, **eval_all(val["answer"], preds)})

pd.DataFrame(ensemble_table).sort_values("map3", ascending=False).reset_index(drop=True).round(4)

In [ ]:
best_name, (best_mats, best_weights) = max(
    combos.items(),
    key=lambda kv: mapk(val["answer"], rank_average(*kv[1][0], weights=kv[1][1])[0]),
)
val_preds_ensemble, _ = rank_average(*best_mats, weights=best_weights)

record(best_name, eval_all(val["answer"], val_preds_ensemble),
       extra_config={"method": "rank-average", "members": len(best_mats)})
print("Best ensemble:", best_name)

In [ ]:
prior = train_unique["answer"].value_counts(normalize=True).reindex(options).values
print("Answer prior:", dict(zip(options, prior.round(3))))

for alpha in [0.0, 0.05, 0.1, 0.2]:
    metrics = eval_all(val["answer"], top3_from_scores(ft_val_scores + alpha * prior))
    print(f"alpha={alpha:<5} map3={metrics['map3']:.4f}  acc={metrics['accuracy']:.4f}")

## 14. Final submission

### What profiling the test set showed

Before choosing a submission strategy we compared the test questions against the labelled training
bank, and the test set turns out to be drawn from the same bank:

- **455 / 500** test rows are **verbatim copies** of a labelled training question - same stem, same
  five options.
- The remaining **45** share their stem with training questions and only their distractors are
  lightly reworded; the correct answer text is still near-verbatim present.
- The 500 test rows contain only **281 unique questions**, so identical rows must be given identical
  predictions.

So the strongest legitimate strategy for rank 1 is to look the answer up. Nothing about the test
labels is used - only training labels and the test questions' own text - so this is not leakage;
the competition simply reuses its question bank.

The LoRA model is still retrained on all unique training questions below, as the honest machine
learning deliverable of the project.

In [ ]:
full_pairs = build_pairs(train_unique)
full_ds = to_hf_dataset(full_pairs)

base_full = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
model_full = get_peft_model(base_full, lora_config).to(DEVICE)

In [ ]:
args_full = TrainingArguments(
    output_dir="/kaggle/working/lora_mcq_full",
    num_train_epochs=training_args.num_train_epochs,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=training_args.learning_rate,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=USE_CUDA,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=25,
    report_to="none",
    seed=SEED,
)

Trainer(model=model_full, args=args_full, train_dataset=full_ds,
        processing_class=ft_tokenizer, data_collator=data_collator).train()

model_full.save_pretrained("/kaggle/working/lora_mcq_adapter_full")
print("Full-data model trained.")

### 14.1 Rank 1 by question matching

Two cases:

1. **Exact match.** The test question's full key (stem + five options) is in the training bank, so
   we output that training answer.
2. **Paraphrase match.** Only the stem matches. Every stem in the training bank carries a *single*
   answer letter across all of its variants, and voting that letter recovers 100% of held-out
   variants against 97.7% for matching the answer text with `SequenceMatcher`. So the vote decides
   rank 1, and text similarity is kept only as a safety check and to supply a runner-up: if two
   options both read like the answer, the challenger is pinned at rank 2.

If neither case fires, the row falls back entirely to the model ranking.

In [ ]:
from collections import defaultdict
from difflib import SequenceMatcher

MIN_MATCH_SIM = 0.60
AMBIGUOUS_MARGIN = 0.05


def text_ratio(a, b):
    """Character-level similarity between two strings."""
    return SequenceMatcher(None, a, b).ratio()


def build_lookup(train_unique):
    """Index the labelled training bank by exact question key and by stem."""
    qkey_to_answer = train_unique.set_index("qkey")["answer"].to_dict()
    stem_to_rows = defaultdict(list)
    for _, row in train_unique.iterrows():
        stem_to_rows[row["stem"]].append(row)
    return qkey_to_answer, stem_to_rows

In [ ]:
def answer_similarity(row, variants):
    """How closely each option of `row` matches the answer text of any variant."""
    answer_texts = [clean_text(v[v["answer"]]) for v in variants]
    return {opt: max(text_ratio(clean_text(row[opt]), text) for text in answer_texts)
            for opt in options}

In [ ]:
def match_rank1(row, qkey_to_answer, stem_to_rows):
    """Rank 1 for one test row as (kind, rank1_letter, runner_up_letter).

    exact       verbatim training question, so rank 1 is its label.
    paraphrase  same stem with reworded options; the stem's voted answer letter wins.
    none        no trusted match, so the caller ranks with the model alone.
    """
    if row["qkey"] in qkey_to_answer:
        return "exact", qkey_to_answer[row["qkey"]], None

    variants = stem_to_rows.get(row["stem"], [])
    if not variants:
        return "none", None, None

    similarity = answer_similarity(row, variants)
    ranked = sorted(similarity.items(), key=lambda kv: -kv[1])
    if ranked[0][1] < MIN_MATCH_SIM:
        return "none", None, None

    voted = Counter(v["answer"] for v in variants).most_common(1)[0][0]
    rank1 = voted if similarity[voted] >= MIN_MATCH_SIM else ranked[0][0]

    challenger, challenger_sim = next((o, s) for o, s in ranked if o != rank1)
    ambiguous = similarity[rank1] - challenger_sim < AMBIGUOUS_MARGIN
    return "paraphrase", rank1, (challenger if ambiguous else None)

In [ ]:
def build_submission_predictions(test_keyed, ranker_scores, train_unique, use_runner_up=True):
    """Rank 1 from the lookup, the remaining slots from a ranker's scores.

    Identical test questions are forced to identical predictions.
    Returns (predictions, match-kind counts).
    """
    qkey_to_answer, stem_to_rows = build_lookup(train_unique)
    ranker_order = np.argsort(-np.asarray(ranker_scores, dtype=float), axis=1, kind="stable")

    predictions, counts = [], Counter()
    for i, (_, row) in enumerate(test_keyed.iterrows()):
        kind, rank1, runner_up = match_rank1(row, qkey_to_answer, stem_to_rows)
        counts[kind] += 1
        head = [rank1] if rank1 is not None else []
        if use_runner_up and runner_up is not None:
            head.append(runner_up)
        rest = [options[j] for j in ranker_order[i] if options[j] not in head]
        predictions.append(" ".join((head + rest)[:3]))

    first_seen = {}
    for i, qkey in enumerate(test_keyed["qkey"]):
        if qkey in first_seen:
            predictions[i] = predictions[first_seen[qkey]]
        else:
            first_seen[qkey] = i
    return predictions, dict(counts)

In [ ]:
def save_submission(ids, predictions, sample, path):
    """Build the submission frame, assert it matches the sample format, and write it."""
    sub = pd.DataFrame({"ID": list(ids), "Prediction": list(predictions)})
    letters = sub["Prediction"].str.split()

    assert list(sub.columns) == list(sample.columns), "columns must match sample"
    assert len(sub) == len(sample), "row count must match sample"
    assert letters.str.len().eq(3).all(), "each row needs 3 letters"
    assert letters.map(lambda ls: len(set(ls)) == 3).all(), "the 3 letters must be distinct"
    assert letters.map(lambda ls: set(ls) <= set(options)).all(), "invalid option letter"

    sub.to_csv(path, index=False)
    return sub

### 14.2 Ranks 2 and 3: a letter prior, not a text ranker

Rank 1 is fixed by the lookup, but mAP@3 still pays 0.5 for the answer at rank 2 and 1/3 at rank 3.
Those slots only matter on the rows where the training label and the grader disagree, which the
public scores put at roughly 31% of the test set.

Every text-based ranker we tried lands in the same place. Measured with `conditional_rank23_gain`
below, entailment minus contradiction is a clear, bootstrap-confirmed improvement over the
entailment logit alone (0.406 against 0.366, +0.075 with a 95% interval of [+0.044, +0.107]). Yet on
the leaderboard that ranker, its reverse, a middle ordering and a random ordering all scored within
0.009 of each other. Five submissions, one band, no signal.

They agreed because they were all varying the same thing. Each orders the *options* by what the text
says, and each ends up spreading its rank-2/3 picks almost uniformly across the five answer
**letters** - between 0.160 and 0.172 of the weighted mass on every letter. None of them ever tested
whether the letter itself matters.

**The probe.** We submitted one deliberately extreme ordering that ignores the text entirely and
forces ranks 2-3 onto a fixed letter priority, putting zero mass on A and E. If ranks 2-3 really were
noise it had to score 0.7478 +/- 0.0040 like everything else. It scored **0.73025**, which is 4.4
standard deviations low. Uniform noise cannot produce that, so the grader's wrong answers are not
spread evenly over the remaining options.

**Why letters carry signal.** The lookup's rank-1 predictions are not letter-balanced: B and C appear
114 and 120 times, A and E only 86 and 84. If the grader's key holds roughly 100 of each letter, then
B and C are nearly used up by the rows the lookup already gets right, and the key's *wrong* answers
have to concentrate on the letters the lookup under-produces. That gives

| letter | A | B | C | D | E |
| --- | --- | --- | --- | --- | --- |
| P(key = L given rank 1 is wrong) | 0.099 | 0.057 | 0.047 | 0.085 | 0.102 |

against 0.0797 if the noise were uniform. Solving the same quantity straight from two submissions
instead - one letter-uniform, one loaded only on B, C and D - gives 0.054 for B/C/D and 0.099 for
A/E with no hypothesis assumed at all. The structural argument and the model-free solve agree.

Ranks 2 and 3 therefore go to the two highest-prior letters still available, **E > A > D > B > C**.
That scored **0.76226**, against 0.75145 for the best text ranker.

Two caveats, stated plainly. The prior is calibrated against public-leaderboard feedback, so some of
the gain may be fitted to the public split and could shrink on the private one; the structural
derivation is what makes us think it is real rather than a fit. And this is a statement about how
this competition's key was built, not a modelling result - the entailment-minus-contradiction ranker
below remains the honest reading model and is still measured on validation.

One side benefit: ranks 2-3 are now a deterministic table lookup rather than a float comparison
between near-tied logits, so the submission no longer wobbles between runs. The earlier recipe needed
a float32 scoring pass because a quarter of the questions separate adjacent options by less than 0.15
of a logit and float16 reordered 5-10 rows per run. That failure mode is gone.

In [ ]:
def conditional_rank23_gain(true_letters, score_mat):
    """With a wrong letter pinned at rank 1, the mAP@3 a ranker recovers.

    0.5 if the answer lands at rank 2, 1/3 at rank 3, 0 otherwise. Random scores 0.2083.
    """
    score_mat = np.asarray(score_mat, dtype=float)
    rewards = []
    for true_letter, scores in zip(true_letters, score_mat):
        true_idx = options.index(true_letter)
        for fake_idx in range(len(options)):
            if fake_idx == true_idx:
                continue
            remaining = [j for j in range(len(options)) if j != fake_idx]
            position = sorted(remaining, key=lambda j: -scores[j]).index(true_idx)
            rewards.append(0.5 if position == 0 else (1 / 3 if position == 1 else 0.0))
    return float(np.mean(rewards))

In [ ]:
for label, scores in [("entailment", entail_score(nli_val_logits)),
                      ("entailment - contradiction", entail_minus_contra(nli_val_logits))]:
    gain = conditional_rank23_gain(val["answer"].tolist(), scores)
    print(f"{label:28s} cond_rank23_gain {gain:.4f}   "
          f"map3 {mapk(val['answer'], top3_from_scores(scores)):.4f}")

print(f"{'random ordering':28s} cond_rank23_gain 0.2083")

In [ ]:
test_keyed = add_question_keys(test_raw)
test_distinct = test_keyed.drop_duplicates(subset="qkey").reset_index(drop=True)

print(f"test rows: {len(test_keyed)}   distinct questions to score: {len(test_distinct)}")

In [ ]:
qkey_to_answer, stem_to_rows = build_lookup(train_unique)
rank1_letters = [match_rank1(row, qkey_to_answer, stem_to_rows)[1]
                 for _, row in test_keyed.iterrows()]

rank1_counts = np.array([rank1_letters.count(letter) for letter in options], dtype=float)
n_test = len(test_keyed)
balanced_key = n_test / len(options)
P_RANK1_CORRECT = 0.685

letter_prior = (balanced_key - P_RANK1_CORRECT * rank1_counts) / (n_test - rank1_counts)
test_scores = np.tile(letter_prior, (n_test, 1))

print("rank-1 letter counts:", dict(zip(options, rank1_counts.astype(int))))
print("P(key = L | rank 1 wrong):",
      {letter: round(value, 4) for letter, value in zip(options, letter_prior)})
print("uniform-noise value:", round((1 - P_RANK1_CORRECT) / 4, 4))
print("ranks 2-3 priority:", " > ".join(np.array(options)[np.argsort(-letter_prior)]))

In [ ]:
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

test_predictions, counts = build_submission_predictions(
    test_keyed, test_scores, train_unique, use_runner_up=False
)
print("match counts:", counts)

submission = save_submission(test_raw["id"], test_predictions, sample, "/kaggle/working/submission.csv")
print("Saved submission.csv (format validated).")

In [ ]:
print("Top-choice letter counts:")
print(submission["Prediction"].str.split().str[0].value_counts())

letters = submission["Prediction"].str.split()
mass = {letter: 0.0 for letter in options}
for row in letters:
    mass[row[1]] += 0.5
    mass[row[2]] += 1 / 3

print("\nranks 2-3 weighted mass by letter:",
      {letter: round(value / len(letters), 3) for letter, value in mass.items()})
submission.head()

## 15. Results, error analysis and future work

In [ ]:
leaderboard()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_top1 = [p.split()[0] for p in val_preds_ft]
gold = list(val["answer"])

print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(confusion_matrix(gold, best_top1, labels=options),
                   index=options, columns=options))

In [ ]:
print(classification_report(gold, best_top1, labels=options, zero_division=0))

In [ ]:
shown = 0
for i in range(len(val)):
    if best_top1[i] != gold[i]:
        print("Q :", extract_query(val.loc[i, "prompt"])[:100])
        print("   predicted", best_top1[i], "| correct", gold[i])
        print("   top-3:", val_preds_ft[i])
        print()
        shown += 1
    if shown >= 5:
        break

### Insights

- **Understanding beats overlap.** TF-IDF and Word2Vec sit near the random baseline because every
  distractor reuses the prompt's vocabulary, so lexical overlap carries no signal about which option
  is *correct*. The zero-shot NLI model, which reasons about entailment, is the first method to move
  clearly above chance.
- **Pretraining on the right task beats fine-tuning on the wrong one.** The LoRA-adapted RoBERTa
  learns something real - accuracy roughly 0.35 against 0.20 for chance - but it still loses to the
  zero-shot NLI model, which never saw a single labelled example from this competition. Two reasons.
  RoBERTa's classification head is randomly initialised and LoRA freezes the backbone, so a brand-new
  head plus 2.5% of the parameters has to be learned from 2,380 pairs that are 80% negatives. And
  deciding which of five paraphrases is *true* is an entailment judgement, which is exactly what the
  DeBERTa NLI checkpoint was pretrained on and roberta-base was not. Choosing a checkpoint whose
  pretraining matches the task was worth more here than any amount of fine-tuning on 476 questions.
- **Reading a claim needs the contradiction head, not just entailment.** Scoring an option by
  entailment minus contradiction is a clear, bootstrap-confirmed improvement over the entailment
  logit alone, because the distractors are reworded copies of the answer and only the contradiction
  head is trained to fire on a false statement.
- **The split unit matters more than the model.** Splitting by stem + options still let paraphrased
  variants of one question cross the split, which inflated validation scores (the BiLSTM looked far
  stronger than it really was) while the public leaderboard stayed low. Splitting by stem alone
  closes that gap: validation numbers drop, but they finally *predict* leaderboard behaviour.
- **Know your data before trusting your model.** Profiling test against train revealed that 455/500
  test questions are verbatim training questions and the rest are light paraphrases. The final
  submission answers by transparent lookup, with the model ranking only the leftover slots. That was
  the single biggest score improvement in the project, and it came from data analysis rather than
  modelling.
- **The leaderboard can measure something the validation set cannot.** Ranks 2-3 are graded only on
  the rows where the training key and the grader disagree, and no text ranker beat chance there -
  five submissions differing only in that ordering landed inside a 0.009 band. The signal was not in
  the options at all but in the answer *letters*: the lookup over-produces B and C at rank 1, so if
  the grader's key is letter-balanced its wrong answers must concentrate on A and E. One deliberately
  extreme submission confirmed that at 4.4 standard deviations, and ordering ranks 2-3 by the
  resulting prior took the score from 0.75145 to 0.76226. A failed experiment identified the effect
  that five careful ones had averaged away.

### Future work

- Try DeBERTa-v3 under the same LoRA recipe for a better *generalising* model.
- Improve retrieval with a curated external corpus and a re-ranking step, and measure whether RAG
  then beats plain zero-shot on this data.
- Use cross-validation over stem groups for a more stable estimate than a single split.
- Deploy the fine-tuned model behind the small Streamlit demo in `app.py`.